In [46]:
# Mengimpor library pandas untuk manipulasi dan analisis data tabel
import pandas as pd

# Membaca dataset training (train.csv) dan dataset testing (test.csv) dari file CSV
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Menampilkan 5 baris pertama dari data training dan testing untuk verifikasi struktur awal
print("5 Baris Pertama Data Training:")
print(train_df.head())
print("\n5 Baris Pertama Data Testing:")
print(test_df.head())

# Memeriksa jumlah nilai kosong (null/NaN) pada setiap kolom di data training
print("\nJumlah Nilai Kosong pada Data Training:")
print(train_df.isnull().sum())


5 Baris Pertama Data Training:
   employee_id         department     region         education gender  \
0        65438  Sales & Marketing   region_7  Master's & above      f   
1        65141         Operations  region_22        Bachelor's      m   
2         7513  Sales & Marketing  region_19        Bachelor's      m   
3         2542  Sales & Marketing  region_23        Bachelor's      m   
4        48945         Technology  region_26        Bachelor's      m   

[5 rows x 13 columns]
   employee_id         department  ... awards_won? avg_training_score
0         8724         Technology  ...           0                 77
1        74430                 HR  ...           0                 51
2        72255  Sales & Marketing  ...           0                 47
3        38562        Procurement  ...           0                 65
4        64486            Finance  ...           0                 61

   length_of_service  awards_won?  avg_training_score  is_promoted  
0                 

In [47]:
# Fungsi untuk melakukan prapemrosesan data (preprocessing)
def preprocess(df):
    # Menyalin dataframe agar tidak merubah dataframe asli
    df = df.copy()

    # Mengisi nilai kosong pada kolom 'education' dengan modus (nilai paling sering muncul)
    mode_education = df['education'].mode()[0]
    df['education'] = df['education'].fillna(mode_education)

    # Mengisi nilai kosong pada kolom 'previous_year_rating' dengan median
    median_rating = df['previous_year_rating'].median()
    df['previous_year_rating'] = df['previous_year_rating'].fillna(median_rating)

    # Menentukan kolom-kolom kategorikal yang perlu dikonversi menjadi numerik
    categorical_columns = [
        'department',
        'region',
        'education',
        'gender',
        'recruitment_channel'
    ]

    # Mengonversi kolom kategorikal menjadi representasi angka menggunakan pd.factorize()
    for col in categorical_columns:
        df[col] = pd.factorize(df[col])[0]

    return df

# Menjalankan fungsi prapemrosesan pada data training dan testing
train_df = preprocess(train_df)
test_df = preprocess(test_df)

# Memverifikasi kembali bahwa tidak ada lagi nilai kosong setelah prapemrosesan
print("Nilai Kosong Data Training Setelah Preprocessing:")
print(train_df.isnull().sum())
print("\nNilai Kosong Data Testing Setelah Preprocessing:")
print(test_df.isnull().sum())


Nilai Kosong Data Training Setelah Preprocessing:
employee_id             0
department              0
region                  0
education               0
gender                  0
recruitment_channel     0
no_of_trainings         0
age                     0
previous_year_rating    0
length_of_service       0
awards_won?             0
avg_training_score      0
is_promoted             0
dtype: int64

Nilai Kosong Data Testing Setelah Preprocessing:
employee_id             0
department              0
region                  0
education               0
gender                  0
recruitment_channel     0
no_of_trainings         0
age                     0
previous_year_rating    0
length_of_service       0
awards_won?             0
avg_training_score      0
dtype: int64


In [48]:
# =======================================================
# IMPLEMENTASI ALGORITMA DECISION TREE (ID3) DARI NOL
# =======================================================

import numpy as np

# Fungsi untuk menghitung nilai Entropy (mengukur ketidakpastian/kemurnian label target)
def entropy(y):
    # Menemukan kelas unik beserta jumlah frekuensi kemunculannya
    classes, counts = np.unique(y, return_counts=True)

    result = 0
    # Menghitung probabilitas setiap kelas p(x) dan log2-nya
    for count in counts:
        p = count / len(y) # Peluang kemunculan kelas p(x)
        result -= p * np.log2(p) # Rumus Shannon Entropy: -p * log2(p)

    return result

# Fungsi untuk menghitung Information Gain dari suatu fitur/atribut
def information_gain(data, feature, target):
    # Menghitung entropi total dari target sebelum dilakukan pembagian
    total_entropy = entropy(data[target])

    # Mencari nilai-nilai unik dari fitur yang akan diuji
    values = data[feature].unique()

    weighted_entropy = 0
    # Menghitung bobot entropi dari masing-masing subset setelah pembagian
    for value in values:
        subset = data[data[feature] == value] # Subset data dengan nilai fitur tertentu
        weight = len(subset) / len(data) # Bobot probabilitas subset (proporsi data)
        weighted_entropy += (
            weight * entropy(subset[target]) # Akumulasi entropi berbobot
        )

    # Gain = Entropi Sebelum Pembagian - Entropi Setelah Pembagian
    return total_entropy - weighted_entropy

# Fungsi untuk memilih fitur terbaik dengan Information Gain tertinggi
def best_feature(data, features, target):
    best_gain = -1
    best = None
    
    # Mengevaluasi Information Gain untuk setiap fitur
    for feature in features:
        gain = information_gain(
            data,
            feature,
            target
        )

        # Menyimpan fitur jika Information Gain-nya lebih besar dari gain terbaik sebelumnya
        if gain > best_gain:
            best_gain = gain
            best = feature

    return best

# Fungsi rekursif untuk membangun struktur Decision Tree
def build_tree(data, features, target):
    labels = data[target]

    # Base Case 1: Jika semua data dalam subset memiliki kelas target yang sama
    if len(labels.unique()) == 1:
        return labels.iloc[0] # Kembalikan kelas tersebut sebagai daun (leaf)

    # Base Case 2: Jika semua fitur telah digunakan untuk pembagian
    if len(features) == 0:
        return labels.mode()[0] # Kembalikan kelas mayoritas/modus sebagai daun (leaf)

    # Menentukan fitur terbaik untuk pembagian di node saat ini
    root = best_feature(
        data,
        features,
        target
    )

    # Inisialisasi node pohon dengan fitur terbaik sebagai kunci
    tree = {root: {}}

    # Membuat daftar fitur sisa dengan menghapus fitur terpilih saat ini
    remaining = [
        f for f in features
        if f != root
    ]

    # Membangun sub-tree secara rekursif untuk setiap nilai unik dari fitur terpilih
    for value in data[root].unique():
        subset = data[data[root] == value]

        # Jika subset kosong, node diisi dengan kelas mayoritas dari data asal
        if len(subset) == 0:
            tree[root][value] = labels.mode()[0]
        # Jika tidak, bangun cabang pohon baru secara rekursif
        else:
            tree[root][value] = build_tree(
                subset,
                remaining,
                target
            )

    return tree

# Fungsi untuk melakukan prediksi kelas pada baris data baru menggunakan pohon keputusan
def predict(tree, sample):
    # Jika tree bukan berupa dictionary, berarti kita telah mencapai daun (leaf)
    if not isinstance(tree, dict):
        return tree
    
    # Ambil fitur root (pencabangan pertama pada sub-tree ini)
    root = list(tree.keys())[0]

    # Dapatkan nilai fitur dari data sampel yang sedang diuji
    value = sample[root]

    # Jika nilai fitur sampel tidak ditemukan dalam cabang pohon keputusan
    if value not in tree[root]:
        return 0 # Kembalikan kelas default (0) jika tidak ada cabang yang cocok
    
    # Telusuri pohon lebih dalam secara rekursif melalui cabang yang sesuai
    return predict(
        tree[root][value],
        sample
    )


In [15]:
# Membagi dataset menjadi training set (80%) dan testing set (20%) secara acak
# Dilakukan secara manual dengan fungsi pandas .sample() tanpa library scikit-learn
# untuk sepenuhnya menaati aturan larangan penggunaan library Machine Learning eksternal.
train_data = train_df.sample(frac=0.8, random_state=42)
test_data = train_df.drop(train_data.index)

# Menentukan label target
TARGET = "is_promoted"

# Membuat list fitur yang akan digunakan (mengecualikan 'employee_id' dan kolom target)
features = [
    col for col in train_df.columns
    if col not in ["employee_id", TARGET]
]

print(features)

# Memulai pembangunan model pohon keputusan
tree = build_tree(
    train_data,
    features,
    TARGET
)

print(tree)


['department', 'region', 'education', 'gender', 'recruitment_channel', 'no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'awards_won?', 'avg_training_score']
{'avg_training_score': {np.int64(47): {'region': {np.int64(19): {'age': {np.int64(35): np.int64(0), np.int64(45): np.int64(0), np.int64(29): np.int64(0), np.int64(30): np.int64(0), np.int64(42): np.int64(0), np.int64(49): np.int64(0), np.int64(46): np.int64(0), np.int64(48): np.int64(0), np.int64(23): np.int64(0), np.int64(31): np.int64(0), np.int64(36): np.int64(0), np.int64(25): np.int64(0), np.int64(37): {'recruitment_channel': {np.int64(1): np.int64(0), np.int64(0): np.int64(1)}}, np.int64(38): np.int64(0), np.int64(33): np.int64(0), np.int64(27): np.int64(0), np.int64(24): np.int64(0), np.int64(39): np.int64(0), np.int64(34): np.int64(0), np.int64(41): np.int64(0), np.int64(26): np.int64(0), np.int64(47): np.int64(0)}}, np.int64(5): {'age': {np.int64(58): np.int64(0), np.int64(48): np.int64(0), np.int64(4

In [50]:
# Menginisialisasi variabel penghitung prediksi yang cocok/benar
correct = 0

# Mengevaluasi model untuk setiap baris data dalam test_data
for _, row in test_data.iterrows():
    # Memprediksi label target
    pred = predict(tree, row)

    # Mengecek apakah prediksi cocok dengan nilai aktual
    if pred == row['is_promoted']:
        correct += 1

# Menghitung metrik Akurasi (Accuracy)
accuracy = correct / len(test_data)
print(f'Akurasi Model pada Test Data: {accuracy:.2%}')


Akurasi Model pada Test Data: 90.20%


In [51]:
# Menginisialisasi parameter Confusion Matrix
# tp: True Positive, tn: True Negative, fp: False Positive, fn: False Negative
tp = fp = tn = fn = 0

# Iterasi pada test data untuk menghitung performa klasifikasi secara detail
for _, row in test_data.iterrows():

    actual = row["is_promoted"]
    pred = predict(tree, row)

    # Menentukan kategori prediksi berdasarkan nilai aktual dan nilai prediksi
    if actual == 1 and pred == 1:
        tp += 1
    elif actual == 0 and pred == 0:
        tn += 1
    elif actual == 0 and pred == 1:
        fp += 1
    elif actual == 1 and pred == 0:
        fn += 1

# Menampilkan hasil evaluasi Confusion Matrix
print("=== Confusion Matrix ===")
print(f"True Positive (TP)  = {tp}")
print(f"True Negative (TN)  = {tn}")
print(f"False Positive (FP) = {fp}")
print(f"False Negative (FN) = {fn}")


=== Confusion Matrix ===
True Positive (TP)  = 338
True Negative (TN)  = 9550
False Positive (FP) = 437
False Negative (FN) = 637


In [52]:
# Menghitung metrik evaluasi klasifikasi
# Precision: Kemampuan model mendeteksi sampel positif dengan benar
precision = tp / (tp + fp)

# Recall: Kemampuan model mendeteksi seluruh sampel positif yang ada
recall = tp / (tp + fn)

# F1-score: Rata-rata harmonis dari Precision dan Recall
f1 = 2 * precision * recall / (precision + recall)

# Menampilkan seluruh hasil evaluasi kinerja model
print("=== Klasifikasi Metrik ===")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


=== Klasifikasi Metrik ===
Precision : 0.4361
Recall    : 0.3467
F1-score  : 0.3863
